In [4]:
import os
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage
model = ChatOpenAI(
    base_url=os.getenv("DASHSCOPE_API_BASE"),
    api_key=os.environ.get("DASHSCOPE_API_KEY"),
    model="qwen-plus-2025-01-25",
)

关键：  
1. 返回JSON格式
2. 类的属性名都要写清楚，并且值也要做相应约束，以及父子级关系要表述清楚

In [11]:
from pydantic import BaseModel, Field
from typing import Optional, List
from enum import Enum

class Person(BaseModel):
    """人物信息"""
    name: str = Field(description="姓名")
    age: int = Field(description="年龄")
    occupation: str = Field(description="职业")

# 创建支持JSON格式的结构化LLM
structured_llm = model.with_structured_output(Person)

# 修改提示词，明确要求JSON格式输出
prompt_with_json = """
请将以下人物信息提取为JSON格式：张三是一名 30 岁的软件工程师

请返回包含name、age、occupation的JSON格式数据。
"""

result = structured_llm.invoke(prompt_with_json)

print(f"\n返回类型: {type(result)}")
print(f"姓名: {result.name}")
print(f"年龄: {result.age}")
print(f"职业: {result.occupation}")


返回类型: <class '__main__.Person'>
姓名: 张三
年龄: 30
职业: 软件工程师


In [18]:
# ============================================================================
# 示例 2：提取多个对象（列表）
# ============================================================================
class Book(BaseModel):
    """书籍信息"""
    title: str = Field(description="书名")
    author: str = Field(description="作者")
    year: int = Field(description="出版年份")


class BookList(BaseModel):
    """书籍列表"""
    books: List[Book] = Field(description="书籍列表")  # List[Book] 表示包含多个 Book 实例的列表,实现了"一对多"的关系模型
# ✅ 有效数据
valid_data = {
    "books": [
        {"title": "Python编程", "author": "John", "year": 2020},
        {"title": "数据科学", "author": "Alice", "year": 2021}
    ]
}
book_list = BookList(**valid_data)  # 自动验证数据
for book in book_list.books:
    print(f"书名：{book.title},",end = " ")    
    print(f"作者：{book.author},",end = " ")   
    print(f"年份：{book.year}")     

书名：Python编程, 作者：John, 年份：2020
书名：数据科学, 作者：Alice, 年份：2021


In [21]:
structured_llm = model.with_structured_output(BookList)

text = """
    《三体》是刘慈欣 2008 年的科幻小说。
    《流浪地球》也是刘慈欣的作品，2000 年出版。
    《北京折叠》是郝景芳 2012 年的小说。
    """
prompt= f"""
请从以下文本中精确提取所有书籍信息：

文本内容：
{text}

提取要求：
1. 严格识别每本书的以下字段：
   - title（书名）
   - author（作者）
   - year（出版年份）

2. 输出格式要求：
   - 必须返回完整的JSON对象
   - 使用字段名：books, title, author, year
   - 确保年份是整数类型

3. 提取规则：
   - 只提取明确提到的书籍信息
   - 如果信息不完整，请根据上下文合理推断
   - 确保作者和书名对应关系正确

请仔细分析文本并提取所有书籍信息。
"""
print(f"\n文本: {text.strip()}")
result = structured_llm.invoke(prompt)

print(f"\n提取到 {len(result.books)} 本书：")
for i, book in enumerate(result.books, 1):
    print(f"  {i}. 《{book.title}》 - {book.author} ({book.year})")


文本: 《三体》是刘慈欣 2008 年的科幻小说。
    《流浪地球》也是刘慈欣的作品，2000 年出版。
    《北京折叠》是郝景芳 2012 年的小说。

提取到 3 本书：
  1. 《三体》 - 刘慈欣 (2008)
  2. 《流浪地球》 - 刘慈欣 (2000)
  3. 《北京折叠》 - 郝景芳 (2012)


In [26]:
class Address(BaseModel):
    """地址"""
    city: str = Field(description="城市")
    district: str = Field(description="区")

class Company(BaseModel):
    """公司信息"""
    name: str = Field(description="公司名称")
    employee_count: int = Field(description="员工数量")
    address: Address = Field(description="公司地址")

structured_llm = model.with_structured_output(Company)

text="阿里巴巴公司在杭州滨江区，有约 10 万名员工"
prompt=f"""
### 任务
从文本中精确提取公司信息，严格遵循以下数据结构规范。

### 数据结构规范
公司信息必须包含三个顶级字段：
1. "name" (字符串): 公司全称
2. "employee_count" (整数): 员工总数（纯数字，不含单位）
3. "address" (对象): 包含两个子字段
   - "city" (字符串): 城市名称
   - "district" (字符串): 区/县名称

### 处理规则
1. 数值转换：将"10万名"转换为整数 100000
2. 地址解析：将"杭州滨江区"拆分为：
   - city: "杭州"
   - district: "滨江区"
3. 严格遵循JSON Schema，不要添加任何额外字段

### 示例
文本："腾讯公司在深圳南山区，有5万名员工"
正确输出：
{{
  "name": "腾讯公司",
  "employee_count": 50000,
  "address": {{
    "city": "深圳",
    "district": "南山区"
  }}
}}

### 待处理文本
{text}
"""
result = structured_llm.invoke(prompt)

print(f"\n公司名称: {result.name}")
print(f"员工数量: {result.employee_count}")
print(f"地址: {result.address.city} - {result.address.district}")


公司名称: 阿里巴巴公司
员工数量: 100000
地址: 杭州 - 滨江区


**下面代码写得很好，值得学习**

In [5]:
from pydantic import BaseModel, Field
from typing import Optional, List
from enum import Enum
# ============================================================================
# 示例 4：可选字段和默认值
# ============================================================================
class Product(BaseModel):
    """产品信息"""
    name: str = Field(description="产品名称")
    price: float = Field(description="价格")
    description: Optional[str] = Field(None, description="产品描述（可选）")
    stock: int = Field(100, description="库存（默认 100）")

structured_llm = model.with_structured_output(Product)

def extract_product(text: str) -> Product:
    prompt = f"""
提取以下文本中的产品信息，并严格遵守数据结构规范：

### 数据结构规范：
- name: 产品名称
- price: 价格
- description: 产品描述（可选）
- stock: 库存（默认 100）
严格遵守JSON Schema，不要添加额外字段

### 处理文本：
{text}
"""
    return structured_llm.invoke(prompt)

def print_product(p: Product, label: str):
    print(f"\n{label}")
    print(f"  名称: {p.name}")
    print(f"  价格: {p.price}")
    print(f"  描述: {p.description}")
    print(f"  库存: {p.stock}")

# 场景1：完整信息
text1 = "iPhone 15 售价 5999 元，最新款智能手机，库存 50 台"
result1 = extract_product(text1)
print_product(result1, "场景1：完整信息")

# 场景2：缺少描述和库存
text2 = "MacBook Pro 售价 12999 元"
result2 = extract_product(text2)
print_product(result2, "场景2：缺少描述和库存")


场景1：完整信息
  名称: iPhone 15
  价格: 5999.0
  描述: 最新款智能手机
  库存: 50

场景2：缺少描述和库存
  名称: MacBook Pro
  价格: 12999.0
  描述: None
  库存: 100


In [ ]:
# ============================================================================
# 示例 5：枚举类型
# ============================================================================
class Priority(str, Enum):
    """优先级"""
    LOW = "低"
    MEDIUM = "中"
    HIGH = "高"


class Task(BaseModel):
    """任务"""
    title: str = Field(description="任务标题")
    priority: Priority = Field(description="优先级：低/中/高")
    completed: bool = Field(False, description="是否完成")
task = Task(title="修复 bug", priority=Priority.HIGH)

print(task.priority)           
print(task.priority == "高")   # True
print(task.priority == Priority.HIGH)  # True
print(task.priority.name)
print(task.priority.value)

Priority.HIGH
True
True
HIGH
高


In [9]:
structured_llm = model.with_structured_output(Task)

print("\n提示: 完成季度报告，这是紧急任务")
text={"完成季度报告，这是紧急任务"}
prompt=f"""
从{text}文本中抽取信息，并严格遵守以下数据结构：

### 数据结构
-title:任务标题
-priority：任务优先级
-completed:是否完成，只有两个值True/False

### 数据处理原则
-priority的值只有低，中，高
-返回JSON数据格式
"""
result = structured_llm.invoke(prompt)

print(f"\n任务: {result.title}")
print(f"优先级: {result.priority.value}")  # "高"
print(f"完成状态: {result.completed}")


提示: 完成季度报告，这是紧急任务

任务: 完成季度报告
优先级: 高
完成状态: False


In [13]:
# ============================================================================
# 示例 6：实际应用 - 客户信息提取（继承了上面代码的Priority类）
# ============================================================================
class CustomerInfo(BaseModel):
    """客户信息"""
    name: str = Field(description="客户姓名")
    phone: str = Field(description="电话号码")
    email: Optional[str] = Field(None, description="邮箱（可选）")
    issue: str = Field(description="问题描述")
    urgency: Priority = Field(description="紧急程度")

structured_llm = model.with_structured_output(CustomerInfo)

conversation = """
    客服: 您好，请问有什么可以帮助您？
    客户: 我是李明，电话 138-1234-5678，我的订单一直没发货，很着急！
    客服: 好的，我帮您查一下
    """

print(f"\n对话记录:\n{conversation}")
prompt=f"""
从以下客服对话中提取客户信息：\n{conversation}中提取信息，并严格遵循一下数据结构：
### 数据结构：
-name:客户名字
-phone:客户电话号码
-email:邮箱地址或者'未提供'
-issue:问题描述
-urgency：紧急程度，其值只有"低","中","高"

返回值必须是JSON格式
"""
result = structured_llm.invoke(prompt)

print("\n提取结果：")
print(f"  客户: {result.name}")
print(f"  电话: {result.phone}")
print(f"  邮箱: {result.email} ")
print(f"  问题: {result.issue}")
print(f"  紧急程度: {result.urgency.value}")


对话记录:

    客服: 您好，请问有什么可以帮助您？
    客户: 我是李明，电话 138-1234-5678，我的订单一直没发货，很着急！
    客服: 好的，我帮您查一下
    

提取结果：
  客户: 李明
  电话: 138-1234-5678
  邮箱: 未提供 
  问题: 订单一直没发货
  紧急程度: 高


In [14]:
# ============================================================================
# 示例 7：实际应用 - 产品评论分析
# ============================================================================
class Sentiment(str, Enum):
    """情感"""
    POSITIVE = "正面"
    NEUTRAL = "中性"
    NEGATIVE = "负面"


class Review(BaseModel):
    """评论"""
    product: str = Field(description="产品名称")
    rating: int = Field(description="评分 1-5")
    sentiment: Sentiment = Field(description="情感倾向")
    pros: List[str] = Field(description="优点列表")
    cons: List[str] = Field(description="缺点列表")
structured_llm = model.with_structured_output(Review)

review_text = """
    这款 iPhone 15 Pro 真的很不错！摄像头非常强大，夜拍效果惊艳。
    钛金属边框手感也很好。但是价格有点贵，而且没有充电器。
    总体来说还是值得购买的，我给 4 分。
    """

print(f"\n评论内容:\n{review_text}")
prompt =f"""
从以下{review_text}中抽取信息，并严格遵循以下数据结构：
### 数据结构
-product：产品名称
-rating: 评分
-sentiment：只有三个值："正面","中性","负面"
-pros：优点列表
-cons:缺点列表
返回格式必须是JSON格式，不要添加额外字段
"""
result = structured_llm.invoke(prompt)

print("\n分析结果：")
print(f"  产品: {result.product}")
print(f"  评分: {result.rating} / 5")
print(f"  情感: {result.sentiment.value}")
print(f"  优点: {', '.join(result.pros)}")
print(f"  缺点: {', '.join(result.cons)}")


评论内容:

    这款 iPhone 15 Pro 真的很不错！摄像头非常强大，夜拍效果惊艳。
    钛金属边框手感也很好。但是价格有点贵，而且没有充电器。
    总体来说还是值得购买的，我给 4 分。
    

分析结果：
  产品: iPhone 15 Pro
  评分: 4 / 5
  情感: 正面
  优点: 摄像头非常强大, 夜拍效果惊艳, 钛金属边框手感很好
  缺点: 价格有点贵, 没有充电器
